## ReACT design

#### Import libraries

In [ ]:
import os, asyncio
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel
from openai import AsyncOpenAI

load_dotenv(override=True)

#### Define Model

In [ ]:
#model = "gpt-4.1-mini"

# Alternatively, you can use a local model
client = AsyncOpenAI(base_url="http://localhost:11434/v1")
model = OpenAIChatCompletionsModel(model = "gpt-oss",openai_client= client)


#### Define Tools

In [ ]:
# Tool 1: Gene function lookup (mocked for demo)
@function_tool
def lookup_gene_function(gene_name: str) -> str:
    """Agent tool to look up gene function."""
    # In real use, call an API or database
    return f"{gene_name} is a tumor suppressor gene involved in DNA repair."

# Tool 2: PubMed search (mocked for demo)
@function_tool
def search_pubmed(gene_name: str) -> str:
    """Agent tool to search PubMed for latest articles on a gene."""
    # In real use, call PubMed API
    return f"Latest article on {gene_name}: 'Role of {gene_name} in cancer therapy, 2025.'"


#### Define Agents

In [ ]:
react_agent = Agent(
    name="BioReACTAgent",
    instructions=(
        "Given a gene name, reason step by step about what information is needed. "
        "Use available tools to look up the gene's function and find the latest research article. "
        "Alternate between reasoning and acting until the task is complete. "
        "Return a concise report with both the function and the article in 50 words or less."
    ),
    model=model, 
    tools=[lookup_gene_function, search_pubmed],
)



#### Call the Agent

In [ ]:
# Example usage
from agents import trace
with trace("react_agent_pipeline"):
    gene = "BRCA1"
    result = await Runner.run(react_agent, gene)
    print(result.final_output)